In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)

# Builder-level summary
builder_summary = (
    df.groupby("builder_name")
    .agg(
        listings=("builder_name", "size"),
        avg_price_lakh=("price_lakh", "mean"),
        median_price_lakh=("price_lakh", "median"),
        avg_area_sqft=("area_sqft", "mean"),
        median_area_sqft=("area_sqft", "median"),
        avg_rate_per_sqft=("rate_per_sqft", "mean"),
        median_rate_per_sqft=("rate_per_sqft", "median")
    )
    .reset_index()
)

# Keep builders with enough listings for comparison
builder_summary = builder_summary[
    builder_summary["listings"] >= 30
].copy()

# RERA approval share
rera_share = (
    df.assign(
        rera_approved=df["rera_approval"].eq("Approved by RERA")
    )
    .groupby("builder_name")["rera_approved"]
    .mean()
    .mul(100)
    .reset_index(name="rera_approved_pct")
)

# Ready-to-move share
ready_share = (
    df.assign(
        ready_to_move=df["status"].eq("Ready to move")
    )
    .groupby("builder_name")["ready_to_move"]
    .mean()
    .mul(100)
    .reset_index(name="ready_to_move_pct")
)

# Dominant property type
dominant_property_type = (
    df.groupby("builder_name")["flat_type"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index(name="dominant_property_type")
)

# Combine builder-level information
builder_summary = (
    builder_summary
    .merge(rera_share, on="builder_name", how="left")
    .merge(ready_share, on="builder_name", how="left")
    .merge(dominant_property_type, on="builder_name", how="left")
)

# Listing volume
print("Top builders by listing volume:")
print(
    builder_summary
    .sort_values("listings", ascending=False)
    [
        [
            "builder_name",
            "listings",
            "median_price_lakh",
            "median_rate_per_sqft"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# Highest median price
print("\nBuilders with highest median price:")
print(
    builder_summary
    .sort_values("median_price_lakh", ascending=False)
    [
        [
            "builder_name",
            "listings",
            "median_price_lakh",
            "avg_price_lakh",
            "dominant_property_type"
        ]
    ]
    .head(15)
    .to_string(index=False)
)

# Highest median rate
print("\nBuilders with highest median rate per sqft:")
print(
    builder_summary
    .sort_values("median_rate_per_sqft", ascending=False)
    [
        [
            "builder_name",
            "listings",
            "median_rate_per_sqft",
            "avg_rate_per_sqft"
        ]
    ]
    .head(15)
    .to_string(index=False)
)

# Largest average property area
print("\nBuilders with largest average property area:")
print(
    builder_summary
    .sort_values("avg_area_sqft", ascending=False)
    [
        [
            "builder_name",
            "listings",
            "avg_area_sqft",
            "median_area_sqft",
            "median_price_lakh"
        ]
    ]
    .head(15)
    .to_string(index=False)
)

# RERA approval share
print("\nBuilders with highest RERA approval share:")
print(
    builder_summary
    .sort_values("rera_approved_pct", ascending=False)
    [
        [
            "builder_name",
            "listings",
            "rera_approved_pct"
        ]
    ]
    .head(15)
    .to_string(index=False)
)

# Ready-to-move share
print("\nBuilders with highest ready-to-move share:")
print(
    builder_summary
    .sort_values("ready_to_move_pct", ascending=False)
    [
        [
            "builder_name",
            "listings",
            "ready_to_move_pct"
        ]
    ]
    .head(15)
    .to_string(index=False)
)

Top builders by listing volume:
                     builder_name  listings  median_price_lakh  median_rate_per_sqft
   properties for sale in Gurgaon       818              269.5               12222.0
              Property In Gurgaon       810              255.0               12048.5
                             home       803              257.0               12222.0
                        Proptiger       734              397.0               16500.0
                      Sharma Prop       490              450.0               11111.0
                  WADHWA REALTORS       299              414.0               21314.0
               Search My Property       242              383.5               15951.5
                              SBL       198              125.0                6515.0
                     The Assetree       165              771.0               22000.0
              Multi Home Realtors       162              507.5               15000.0
INFRAMANTRA INDIA PRIVATE LIMITED